In [ ]:
import yaml
import glob
import json
import os
from dataclasses import dataclass
from pprint import pprint
from typing import Dict, Optional, List, Any
from slideguard.schemes import FullEvaluation
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.language_models import LanguageModelInput


# Functions

In [2]:
class VLLMChatOpenAI(ChatOpenAI):
    def _get_request_payload(
        self,
        input_: LanguageModelInput,
        *,
        stop: Optional[List[str]] = None,
        **kwargs: Any,
    ) -> dict:
        payload = super()._get_request_payload(input_, stop=stop, **kwargs)
        # max_tokens was deprecated in favor of max_completion_tokens
        # in September 2024 release
        if "max_completion_tokens" in payload:
            payload["max_tokens"] = payload.pop("max_completion_tokens")
        return payload
    

def load_goldens(base_path: str = "../golden") -> Dict[str, dict]:
    golden_files = glob.glob(os.path.join(base_path, "*.yaml"))

    goldens = dict()
    for file in golden_files:
        deck_name, _ = os.path.splitext(os.path.basename(file))
        with open(file, "r") as f:
            evaluation = yaml.safe_load(f)
        goldens[deck_name] = evaluation
    
    return goldens


def load_evaluations(base_path: str = "../slidedecks_test_evaluations") -> Dict[str, FullEvaluation]:
    evaluation_files = glob.glob(os.path.join(base_path, "evaluations_*.json"))

    evaluations = dict()
    for file in evaluation_files:
        deck_name, _ = os.path.splitext(os.path.basename(file))
        deck_name = deck_name.replace("evaluations_", "")
        with open(file, "r") as f:
            evaluation = FullEvaluation.model_validate_json(f.read())
        evaluations[deck_name] = evaluation
    
    return evaluations

In [6]:
goldens = load_goldens()
evaluations = load_evaluations()

print("Goldens:", len(goldens))
print("Evaluations:", len(evaluations))

Goldens: 3
Evaluations: 3


In [7]:
golden2evaluation = dict()
for deck_name, golden in goldens.items():
    if deck_name in evaluations:
        golden2evaluation[deck_name] = (golden, evaluations[deck_name])
    else:
        print(f"No evaluation for {deck_name}")

print(len(golden2evaluation))

3


In [8]:
golden2evaluation.keys()

dict_keys(['1_EN_Kataeva_Thesis', '15_RU_Basilaev_Thesis', '12_EN_Zamiralov_NIR'])

In [9]:
gold, evaluation = golden2evaluation['1_EN_Kataeva_Thesis']

ev_results = dict()

def _convert(el):
    del el['severity']
    return el

for criteria, evaluations in evaluation.deck_evaluations.evaluations.items():
    eval_results = [_convert(el) for el in evaluations['evaluation_results'] if el['severity'] > 2]
    ev_results[criteria.value] = eval_results

pprint(ev_results)

{'deck_research_quality': [{'evaluation_element': 'Scientific rigor of the '
                                                  'research',
                            'evaluation_suggestion': 'The research employs '
                                                     'appropriate scientific '
                                                     'methods and metrics. '
                                                     'However, the '
                                                     'presentation could '
                                                     'elaborate more on the '
                                                     'statistical significance '
                                                     'of the results and '
                                                     'discuss potential '
                                                     'limitations and future '
                                                     'work to further solidify '
                

In [ ]:

system_prompt = """
You are a helpful assistant.
You need to estimate if a golden comment made by a human expert is presented in the set of evaluation comments made by an AI agent. 
The expert's comment may have a different structure and form than the evaluation comment, 
but the essence of the comment should be the same.
AI agent may have many comments in the set, so you need to find at least one comment that is the most similar to the expert's comment.

Answer with "yes" or "no".
"""

human_prompt = """
The comment made by a human expert:
{human_comment}

The set of evaluations comment made by an AI agent:
{ai_comments}
"""
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", human_prompt)
])

llm = VLLMChatOpenAI(
    model="/model",
    temperature=0.1,
    max_completion_tokens=1000,
    max_tokens=1000,
    base_url="http://d.dgx:8082/v1",
    api_key="token-abc123"
)

chain = (
    chat_prompt | llm | StrOutputParser()
)

/home/nikolay/.cache/pypoetry/virtualenvs/slideguard-UU6zwuLG-py3.12/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3639: UserWarning: WARNING! api_base is not default parameter.
                api_base was transferred to model_kwargs.
                Please confirm that api_base is what you intended.
  if await self.run_code(code, result, async_=asy):


In [11]:
str(ev_results['deck_structure_analysis'])

'[{\'evaluation_element\': \'Motivation\', \'evaluation_suggestion\': \'The presentation lacks a dedicated slide for motivation. While some slides touch upon motivational aspects, a clear and separate slide outlining the motivation behind the research is missing.\'}, {\'evaluation_element\': \'Goals\', \'evaluation_suggestion\': "The presentation lacks a dedicated slide for goals. Although the \'Objectives of research\' slide includes a goal, having a separate slide explicitly stating the goals would enhance clarity."}, {\'evaluation_element\': \'Tasks\', \'evaluation_suggestion\': "The presentation lacks a dedicated slide for tasks. While tasks are mentioned in the \'Objectives of research\' slide, a separate slide focusing solely on tasks would improve the structure."}, {\'evaluation_element\': \'Current State\', \'evaluation_suggestion\': \'The presentation lacks a dedicated slide for the current state. Although some slides mention the current state, a separate slide clearly outlini

In [13]:
for comment in gold['evaluation']['deck']['deck_structure_analysis']:
    result = chain.invoke({
        "human_comment": comment,
        "ai_comments": str(ev_results['deck_structure_analysis'])
    })
    print("-" * 100)
    print("Expert comment:", comment['comment'])
    print("Result:", result)

TypeError: Completions.create() got an unexpected keyword argument 'api_base'